In [0]:
!pip install pyextremes

In [0]:
import pandas as pd
import numpy as np
from pyextremes import EVA
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
# Load the data
df = spark.read.table("cor_project.silver.swell_metrics")
df = df.toPandas()


In [0]:
df.describe()

In [0]:
df.shape

In [0]:
# ELIMINAR COLUMNAS INSERVIBLES
df = df.drop(columns = ['id','source_file','ingestion_timestamp','transformation_timestamp', 'wind_cos_direction', 'wind_sin_direction', 'wave_cos_direction', 'wave_sin_direction'])


In [0]:
"""
CALCULO DE POTENCIA 
La gravedad g = 9.81 m/s² y la densidad del agua de mar
ρ = 1025 kg/m³ se resumen en el factor aproximado de 490.
"""

# Potencia de ola (kW/m)
df['wave_power_kW_m'] = (
    490 * (df['wave_height_m'] ** 2) * df['wave_period_s']
) / 1000

In [0]:
df.columns

In [0]:
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.set_index('datetime')
df = df.sort_index()

In [0]:
RETURN_PERIODS = [50, 100]
BOOTSTRAP_ITERATIONS = 1000
R_DECLUSTERING = "72h"
THRESHOLD_QUANTILE = 0.95

def run_pot_analysis(ts: pd.Series, coast_name: str):
    """
    Ejecuta el análisis POT
    """
    try:
        threshold = ts.quantile(THRESHOLD_QUANTILE)
        model = EVA(ts)
        
        # Se obtienen los extremos
        model.get_extremes(method="POT", threshold=threshold, r=R_DECLUSTERING)
        
        # Se ajusta el modelo
        model.fit_model(distribution="genpareto")
        
        # Resultados
        results = {"coast": coast_name}
        for rp in RETURN_PERIODS:
            rl, ci_lower, ci_upper = model.get_return_value(
                return_period=rp,
                return_period_size="365.2425D",
                alpha=0.95,
                n_samples=BOOTSTRAP_ITERATIONS
            )
            results[f"RL_{rp}"] = rl
            results[f"CI_lower_{rp}"] = ci_lower
            results[f"CI_upper_{rp}"] = ci_upper
            
        return results
    except Exception as e:
        print(f"Error en {coast_name}: {e}")
        return {"coast": coast_name, "error": str(e)}

In [0]:
df_clean = df[['coast_name', 'wave_power_kW_m']].copy()

# Listado de costas
coasts = df_clean['coast_name'].unique()

# Ejecución en paralelo usando todos los núcleos disponibles (n_jobs=-1)
results_list = Parallel(n_jobs=-1)(
    delayed(run_pot_analysis)(
        df_clean[df_clean['coast_name'] == coast]['wave_power_kW_m'].sort_index(), 
        coast
    ) 
    for coast in coasts
)

In [0]:
df_clean

In [0]:
results_df = pd.DataFrame(results_list)
print("\n--- Resultados Finales ---")
results_df

In [0]:
df_plot = results_df.sort_values(by='RL_50', ascending=True)

plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

# El eje X es el RL, el eje Y es la costa
plt.errorbar(
    x=df_plot['RL_50'], 
    y=df_plot['coast'], 
    xerr=[df_plot['RL_50'] - df_plot['CI_lower_50'], df_plot['CI_upper_50'] - df_plot['RL_50']],
    fmt='o',            # Punto central
    color='#1f77b4',    
    ecolor='#d62728',   # Rojo para el error 
    elinewidth=2,       
    capsize=5,          
    markersize=8,
    label='RL de 50 años con IC 95%'
)

plt.title('Evaluación de Riesgo: Potencia de Ola (RL 50 años)', fontsize=14, fontweight='bold')
plt.xlabel('Potencia de Ola (kW/m)', fontsize=12)
plt.ylabel('Costa', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)

# Destacar la zona de alta energía
plt.axvline(x=df_plot['RL_50'].mean(), color='gray', linestyle=':', label='Promedio General')

plt.legend()
plt.tight_layout()
plt.show()

In [0]:
df_plot = results_df.sort_values(by='RL_100', ascending=True)

plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

# El eje X es el RL, el eje Y es la costa
plt.errorbar(
    x=df_plot['RL_100'], 
    y=df_plot['coast'], 
    xerr=[df_plot['RL_100'] - df_plot['CI_lower_100'], df_plot['CI_upper_100'] - df_plot['RL_100']],
    fmt='o',            # Punto central
    color='#1f77b4',    
    ecolor='#d62728',   # Rojo para el error 
    elinewidth=2,       
    capsize=5,          
    markersize=8,
    label='RL de 50 años con IC 95%'
)

plt.title('Evaluación de Riesgo: Potencia de Ola (RL 100 años)', fontsize=14, fontweight='bold')
plt.xlabel('Potencia de Ola (kW/m)', fontsize=12)
plt.ylabel('Costa', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)

# Destacar la zona de alta energía
plt.axvline(x=df_plot['RL_100'].mean(), color='gray', linestyle=':', label='Promedio General')

plt.legend()
plt.tight_layout()
plt.show()

In [0]:
profiles = []

for coast in df['coast_name'].unique(): #Itera en el dataframe la columna 'coast_name' solo por cada costa
	df_costa = df[df['coast_name'] == coast] # quedarse solo con las filas de la costa actual
	
	# threshold percentile 95
	threshold = df_costa['wave_power_kW_m'].quantile(0.95) # las filas que superen el percentil 95 de 'wave_power_kW_m'.
	
	# Filtrar filas extremas
	extremes = df_costa[df_costa['wave_power_kW_m'] >= threshold][['coast_name', 'wave_height_m', 'wave_direction_deg', 'wave_period_s','wave_power_kW_m']]
 
	profiles.append(extremes) 
	
df_extremes = pd.concat(profiles)
print(df_extremes.groupby('coast_name').describe())

In [0]:
df_extremes

In [0]:
df_extremes.groupby('coast_name').describe()

In [0]:
# Para cada costa y cada periodo de retorno
for coast in results_df['coast']:
    # Usar la  mediana del periodo de eventos extremos de esa costa
    T_mediana = df_extremes[
        df_extremes['coast_name'] == coast
    ]['wave_period_s'].median()
    
    # Despejar H para RL50 y RL100
    for rp in [50, 100]:
        RL = results_df[results_df['coast'] == coast][f'RL_{rp}'].values[0]
        
        H_estimada = np.sqrt(RL / (0.49 * T_mediana))
        
        print(f"{coast} - RL{rp}: {RL:.1f} kW/m → H estimada: {H_estimada:.2f} m")

In [0]:
# 1. Definir los Bins de Potencia
bins = [0, 50, 100, 200, 500, np.inf]
labels = ['bajo', 'medio', 'alto', 'muy_alto', 'extremo']

# 2.Calcula la mediana del periodo para cada "bin" de potencia en los datos extremos
df_extremes['power_bin'] = pd.cut(df_extremes['wave_power_kW_m'], bins=bins, labels=labels)
period_map = df_extremes.groupby('power_bin')['wave_period_s'].median().to_dict()

# 3. Función para asignar periodo según la potencia estimada (RL)
def get_period_for_power(power_value):
    # Identificar a qué bin pertenece la potencia
    bin_name = pd.cut([power_value], bins=bins, labels=labels)[0]
    return period_map.get(bin_name, df_extremes['wave_period_s'].median()) # Fallback a mediana global

# 4. Cálculo de Rangos de Altura (H) usando RL_lower, RL_mean, RL_upper
for rp in [50, 100]:
    for bound in ['lower', 'mean', 'upper']:
        if bound == 'mean':
            P = results_df[f'RL_{rp}']
        else:
            P = results_df[f'CI_{bound}_{rp}']
            
        #Periodo correlacionado la potencia
        T_asociado = P.apply(get_period_for_power)
        
        # Calcular H
        H_estimada = np.sqrt(P / (0.49 * T_asociado))
        
        results_df[f'H_{rp}_{bound}'] = H_estimada

In [0]:
# Para ver solo las columnas de interés (Altura de ola para 50 y 100 años)
columnas_interes = [
    'coast', 
    'H_50_lower', 'H_50_mean', 'H_50_upper', 
    'H_100_lower', 'H_100_mean', 'H_100_upper'
]

# En Databricks, display() es mejor que print() porque crea una tabla interactiva
display(results_df[columnas_interes])

In [0]:

plt.figure(figsize=(12, 6))

# Dibujamos para el periodo de 100 años

plt.errorbar(
    x=results_df['H_100_mean'], 
    y=results_df['coast'], 
    xerr=[
        results_df['H_100_mean'] - results_df['H_100_lower'], # Distancia al lower
        results_df['H_100_upper'] - results_df['H_100_mean']  # Distancia al upper
    ],
    fmt='o', 
    color='darkblue', 
    ecolor='salmon', 
    capsize=5, 
    label='H_s (100 años) ± IC 95%'
)

plt.title('Altura de Ola Significativa Estimada (Periodo de Retorno 100 años)', fontsize=14)
plt.xlabel('Altura Significativa (m)', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.legend()
plt.show()